In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv(
    "../data/processed/energy_cleaned.csv",
    parse_dates=["datetime"],
    index_col="datetime"
)

df.head()


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
datetime,,,,,,,
2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


In [4]:
df["hour"] = df.index.hour
df["day"] = df.index.day
df["dayofweek"] = df.index.dayofweek
df["month"] = df.index.month


In [5]:
df["lag_1"] = df["Global_active_power"].shift(1)
df["lag_24"] = df["Global_active_power"].shift(24)


In [6]:
df["rolling_mean_6"] = df["Global_active_power"].rolling(window=6).mean()
df["rolling_mean_24"] = df["Global_active_power"].rolling(window=24).mean()


In [7]:
df["reactive_ratio"] = (
    df["Global_reactive_power"] /
    (df["Global_active_power"] + 1e-6)
)


In [8]:
active_norm = (
    df["Global_active_power"] - df["Global_active_power"].min()
) / (
    df["Global_active_power"].max() - df["Global_active_power"].min()
)

reactive_norm = (
    df["Global_reactive_power"] - df["Global_reactive_power"].min()
) / (
    df["Global_reactive_power"].max() - df["Global_reactive_power"].min()
)

df["efficiency_score"] = 100 * (1 - (0.7 * active_norm + 0.3 * reactive_norm))
df["efficiency_score"] = df["efficiency_score"].clip(0, 100)


In [9]:
df["target_next_hour"] = df["Global_active_power"].shift(-1)


In [10]:
feature_cols = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",
    "hour",
    "dayofweek",
    "lag_1",
    "lag_24",
    "rolling_mean_6",
    "rolling_mean_24",
    "reactive_ratio",
    "efficiency_score"
]

X = df[feature_cols]
y = df["target_next_hour"]


In [ ]:
X.to_csv("../data/processed/X_features.csv", index=False)
y.to_csv("../data/processed/y_target.csv", index=False)
